In [0]:
%python
# Databricks notebook source
source_path='/Volumes/finguard/source/fraud_watchlist/source_data/'

# COMMAND ----------

dbutils.fs.ls(source_path)


In [0]:
%python



input_stream = spark.readStream.format("cloudFiles").option("cloudFiles.format",'json')\
                               .option("cloudFiles.schemaLocation", "/Volumes/finguard/source/fraud_watchlist/schema/")\
                               .option("cloudFiles.schemaEvolutionMode", "rescue")\
                               .option("cloudFiles.inferSchema", "true")\
                               .load(source_path)
from pyspark.sql.functions import current_timestamp, col

enriched_layer = input_stream.withColumn("ingest_time", current_timestamp()) \
                             .withColumn("File_Path", col("_metadata.file_path"))

streaming_query=(enriched_layer.writeStream.format("delta")
.outputMode("Append")
.option("checkpointLocation", "/Volumes/finguard/source/fraud_watchlist/checkpoint/")
.trigger(availableNow=True)
.toTable("finguard.bronze.fraud_watchlist_batch_test")
)
                               